# Stage C 03g — fresh 5M deep-adaptive exploratory scale gate

This runs exactly one fresh 5,000,000-valid-base compact deep-adaptive model under the paper-exact recurrence. It registers a source-controlled exploratory amendment, captures checkpoints/logs/live status to Drive, then runs held-out trace/PCA and controlled association analyses. It intentionally does not launch no-memory, frozen, shallow, or replication controls.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='07c7084533b337a227bdc0d98400a8aca4df984c'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
RUN_NAME='c16_deep_adaptive_5m_paper_exact'
BUDGET_BASES=5_000_000; BLOCK_COUNT=4; D_MODEL=128; NUM_HEADS=4; HORIZON=3
CHECKPOINT_EVERY=250; VALIDATION_STREAMS=8; TRACE_MAX_SEGMENTS=256; BEHAVIOR_PAIRS=64


In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount), timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Use a Colab T4-or-better GPU runtime for this run.')
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
AMENDMENT=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/amendments/adaptive_exploration_5m_v1.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
run_dir=Path(DRIVE_ROOT)/'runs'/RUN_NAME
if not AMENDMENT.is_file(): raise FileNotFoundError(f'Missing source-controlled amendment: {AMENDMENT}')
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
source_amendment=json.loads(AMENDMENT.read_text())
drive_amendment=STUDY_ROOT/'amendments'/f"{source_amendment['amendment_id']}.json"
if not drive_amendment.exists():
    subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',source_amendment['amendment_id'],'--rationale',source_amendment['rationale'],'--classification',source_amendment['classification'],'--expected-impact',source_amendment['expected_impact'],'--changes',json.dumps(source_amendment['changes'],sort_keys=True)],check=True)
else:
    print('Drive amendment already recorded:',drive_amendment)
print('Fresh exploratory run directory:',run_dir)


In [ ]:
def run_logged(log_dir,label,command):
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(log_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (log_dir/'FAILED.txt',log_dir/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-16000:])
        raise
def record_once(marker_name,run_id,artifact):
    marker=STUDY_ROOT/'record_markers'/marker_name
    if marker.exists():
        print('Ledger record already exists:',marker)
        return
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier','exploratory','--artifact',str(artifact)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')
def deep_flags():
    return ['--memory-architecture','paper_residual_mlp_v2','--memory-depth','2','--memory-expansion-factor','4','--memory-projection-convolution-kernel','4','--memory-normalize-queries-and-keys','--memory-gate-granularity','per_layer_channel','--memory-recurrence-policy','paper_exact','--memory-surprise-clip-norm','none','--memory-alpha-initial','0.001','--memory-eta-initial','0.9','--memory-theta-initial','0.001','--memory-associative-loss-reduction','sum','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','none','--memory-theta-max','1.0']
if not (run_dir/'hardware_preflight.json').is_file():
    run_logged(run_dir,'hardware_preflight',['seqtrainer-titans-stage-c-hardware-preflight','--require','T4','--output',str(run_dir/'hardware_preflight.json')])
if not (run_dir/'validation.json').is_file():
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',str(dataset),'--run-dir',str(run_dir),'--memory-mode','adaptive','--horizon',str(HORIZON),'--batch-size','1','--max-valid-bases',str(BUDGET_BASES),'--checkpoint-every',str(CHECKPOINT_EVERY),'--learning-rate','3e-5','--gradient-clip-norm','0.5','--validation-streams',str(VALIDATION_STREAMS),'--activation','float32','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens','4',*deep_flags(),'--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--run-id','adaptive_exploration_5m']
    run_logged(run_dir,'train_deep_adaptive_5m',command)
else:
    print('Completed validation already exists; the train command is skipped to avoid an accidental rerun.')
checkpoint=run_dir/'latest.pt'
if not checkpoint.is_file(): raise FileNotFoundError(f'Expected completed/resumable checkpoint: {checkpoint}')
if not (run_dir/'MODEL_ARCHITECTURE.txt').is_file():
    run_logged(run_dir,'architecture_deep_adaptive_5m',['seqtrainer-titans-stage-c-architecture','--checkpoint',str(checkpoint),'--output-dir',str(run_dir)])
print((run_dir/'MODEL_ARCHITECTURE.txt').read_text())


In [ ]:
# Analysis is deliberately downstream of the completed adaptive-only run. No controls are launched here.
trace_dir=run_dir/'memory_trace_val'
if not (trace_dir/'memory_trace.json').is_file():
    run_logged(trace_dir,'memory_trace_5m',['seqtrainer-titans-stage-c-memory-trace','--dataset-dir',str(dataset),'--checkpoint',str(checkpoint),'--output-dir',str(trace_dir),'--split','val','--memory-mode','adaptive','--max-streams',str(VALIDATION_STREAMS),'--max-segments',str(TRACE_MAX_SEGMENTS),'--device','cuda','--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--run-id','adaptive_exploration_5m_trace'])
record_once('adaptive_exploration_5m_trace.json','adaptive_exploration_5m_trace',trace_dir)
behavior=run_dir/'controlled_memory_behavior.json'
if not behavior.is_file():
    run_logged(run_dir,'controlled_memory_behavior_5m',['seqtrainer-titans-stage-c-memory-behavior','--checkpoint',str(checkpoint),'--output',str(behavior),'--pairs',str(BEHAVIOR_PAIRS),'--device','cuda','--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--run-id','adaptive_exploration_5m_behavior'])
record_once('adaptive_exploration_5m_behavior.json','adaptive_exploration_5m_behavior',behavior)
record_once('adaptive_exploration_5m_train.json','adaptive_exploration_5m',run_dir)
subprocess.run(['seqtrainer-titans-stage-c-study','verify','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
validation=json.loads((run_dir/'validation.json').read_text())
trace=json.loads((trace_dir/'memory_trace.json').read_text())
probe=json.loads(behavior.read_text())
print('5M held-out validation:')
print(json.dumps({key:validation.get(key) for key in ('bits_per_base','loss_per_token','token_accuracy','memory_update_norm_mean','surprise_norm_mean','state_drift_norm_mean')},indent=2,sort_keys=True))
print('5M held-out trace:')
print(json.dumps({key:trace[key] for key in ('segments','streams','mean_bits_per_base','mean_memory_update_norm','mean_surprise_norm','pca_explained_variance','pc1_gc_correlation','pc1_stream_position_correlation')},indent=2,sort_keys=True))
print('5M controlled association:',json.dumps({'all_blocks_improve_immediately':probe['all_blocks_improve_immediately'],'blocks_with_positive_delayed_margin':probe['blocks_with_positive_delayed_margin']},indent=2))
print('Review training_history.json, validation.json, memory_trace_val/memory_pca.svg, and controlled_memory_behavior.json before proposing any matched no-memory control.')
print('Live status during training:',run_dir/'LIVE_STATUS.json')
